# Aalborg açık araştırma verisi
Önce repo kökünde `python scripts/run_aalborg.py` çalıştırılır. Hazır snapshot'ları inceliyoruz.
Kaynak: Schaffer ve diğerleri (2025), [10.5281/zenodo.15180141](https://doi.org/10.5281/zenodo.15180141), CC BY 4.0. Seri interpolasyon ve günlük minimum çıkarımı içerir; ham tüketim veya kaçak gerçeği değildir.

In [ ]:
import json
from decimal import Decimal
from pathlib import Path

import pyarrow.parquet as pq
from urban_water.db import PROJECT_ROOT

reference_root = PROJECT_ROOT / "var/research/aalborg/v1"
report = json.loads((reference_root / "import-report.json").read_text(encoding="utf-8"))
audit = json.loads((reference_root / "source-audit.json").read_text(encoding="utf-8"))
{
    key: report[key]
    for key in [
        "source_rows",
        "households",
        "total_liters",
        "value_semantics",
        "receipt_basis",
        "trained",
    ]
}

## Günlük çıktı
Yerel günler Europe/Copenhagen ile hesaplanır. İlk ve son gün kısmi olduğundan günlük hedef seçiminde ayrıca filtrelenir. observed_liters kesin ondalık metindir.

In [ ]:
daily = pq.read_table(Path(report["outputs"]["day"]))
daily.slice(0, 5).to_pylist()

In [ ]:
hourly = pq.read_table(Path(report["outputs"]["hour"]))
day_total = sum(map(Decimal, daily["observed_liters"].to_pylist()))
hour_total = sum(map(Decimal, hourly["observed_liters"].to_pylist()))
assert day_total == hour_total == Decimal(audit["source_total_liters"])
{"daily_rows": daily.num_rows, "hourly_rows": hourly.num_rows, "total_liters": str(day_total)}

## Takvim ve kapsam
Upstream interpolasyon nedeniyle dolu saat ızgarası ham ölçüm kapsamasını kanıtlamaz; coverage unknown kalır.

In [ ]:
households = audit["buildings"]
{
    "complete_household_days": sum(row["complete_calendar_days"] for row in households.values()),
    "partial_household_days": sum(len(row["partial_calendar_days"]) for row in households.values()),
    "first_household_calendar": next(iter(households.values())),
}

## Sonraki deney
Tam günler seçilip zaman sıralı eğitim/doğrulama/test ayrımı kurulacak. Günlük minimum çıkarımı gelecekteki saatlerden etkilendiği için ilk deney işlenmiş seri üzerinde geriye dönük bir karşılaştırma olarak tanımlanmalıdır. Doluluk etiketleri özelliklere eklenmedi. Arşiv alımı, 2022 sağlayıcı teslimat saati değildir.